# X-Hour 3: Embeddings Workshop

**CS 165: Natural Language Processing**  
**Week 3 - Thursday X-Hour**

---

## Learning Objectives

By the end of this session, you will:
1. Understand the concept of word embeddings and why they're useful
2. Implement classic dimensionality reduction methods (LSA, LDA)
3. Compute and analyze word2vec embeddings
4. Visualize high-dimensional embeddings using UMAP
5. Compare different embedding methods on real data
6. Understand semantic relationships captured by embeddings

## Setup

In [ ]:
# Install required packages (for Colab)
!pip install -q scikit-learn pandas numpy matplotlib seaborn gensim umap-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
import umap
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("✓ All imports successful!")

## Introduction: Why Embeddings?

Last week we used **sparse representations** (BoW, TF-IDF):
- High dimensional (vocab size)
- Sparse (mostly zeros)
- No semantic similarity

**Embeddings** are **dense, low-dimensional representations** that capture semantic meaning:
- Low dimensional (50-300 dims)
- Dense (all non-zero)
- Similar words have similar vectors

Example:
- "king" - "man" + "woman" ≈ "queen"
- Similar words cluster together

## Part 1: Loading and Preparing Data

We'll use the 20 Newsgroups dataset again, but this time we'll focus on learning semantic representations.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

# Load a larger subset for better embeddings
categories = [
    'sci.space',
    'sci.med',
    'rec.sport.hockey',
    'rec.sport.baseball',
    'talk.politics.misc',
    'talk.religion.misc',
    'comp.graphics',
    'comp.os.ms-windows.misc'
]

print("Loading 20 Newsgroups dataset...")
newsgroups = fetch_20newsgroups(
    subset='all',  # Use all data for better embeddings
    categories=categories,
    shuffle=True,
    random_state=42,
    remove=('headers', 'footers', 'quotes')
)

documents = newsgroups.data
labels = newsgroups.target
label_names = newsgroups.target_names

print(f"\n✓ Loaded {len(documents)} documents")
print(f"✓ Categories: {len(categories)}")
print(f"\nCategories: {label_names}")

In [ ]:
# Preprocess documents for word-level analysis
def preprocess_for_embeddings(text):
    """
    Simple preprocessing for embeddings.
    Tokenizes and lowercases, removes very short words.
    """
    tokens = simple_preprocess(text, deacc=True)  # Remove accents, lowercase
    # Remove very short tokens
    tokens = [t for t in tokens if len(t) > 2]
    return tokens

# Tokenize all documents
print("Tokenizing documents...")
tokenized_docs = [preprocess_for_embeddings(doc) for doc in documents]

# Build vocabulary
vocab = set()
for doc in tokenized_docs:
    vocab.update(doc)

print(f"✓ Vocabulary size: {len(vocab)} unique words")
print(f"✓ Average document length: {np.mean([len(doc) for doc in tokenized_docs]):.1f} words")

## Part 2: Classic Embeddings - LSA (Latent Semantic Analysis)

LSA uses **Singular Value Decomposition (SVD)** to reduce dimensionality of TF-IDF matrix:

$$X \approx U_k \Sigma_k V_k^T$$

where $k$ is the desired embedding dimension.

- Projects documents and words into same semantic space
- Captures co-occurrence patterns
- Also called Latent Semantic Indexing (LSI)

In [ ]:
# Create TF-IDF matrix
print("Creating TF-IDF matrix...")
tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.5,
    stop_words='english'
)

tfidf_matrix = tfidf.fit_transform(documents)
feature_names = tfidf.get_feature_names_out()

print(f"✓ TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"  {tfidf_matrix.shape[0]} documents")
print(f"  {tfidf_matrix.shape[1]} features")

In [ ]:
# Apply LSA (SVD)
n_components = 100

print(f"Applying LSA with {n_components} components...")
lsa = TruncatedSVD(n_components=n_components, random_state=42)
doc_embeddings_lsa = lsa.fit_transform(tfidf_matrix)
word_embeddings_lsa = lsa.components_.T  # Transpose to get word embeddings

print(f"\n✓ Document embeddings shape: {doc_embeddings_lsa.shape}")
print(f"✓ Word embeddings shape: {word_embeddings_lsa.shape}")
print(f"✓ Explained variance: {lsa.explained_variance_ratio_.sum():.3f}")

In [ ]:
# Plot explained variance
plt.figure(figsize=(10, 5))
plt.plot(np.cumsum(lsa.explained_variance_ratio_))
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('LSA: Explained Variance by Number of Components')
plt.grid(True)
plt.show()

print(f"First 10 components explain {lsa.explained_variance_ratio_[:10].sum():.1%} of variance")
print(f"First 50 components explain {lsa.explained_variance_ratio_[:50].sum():.1%} of variance")

In [ ]:
# Helper function to find similar words
def find_similar_words_lsa(word, word_embeddings, feature_names, top_n=10):
    """
    Find words most similar to the query word using LSA embeddings.
    """
    if word not in feature_names:
        return f"Word '{word}' not in vocabulary"
    
    # Get word index
    word_idx = np.where(feature_names == word)[0][0]
    word_vec = word_embeddings[word_idx].reshape(1, -1)
    
    # Compute similarities
    similarities = cosine_similarity(word_vec, word_embeddings)[0]
    
    # Get top similar words (excluding the word itself)
    top_indices = np.argsort(similarities)[::-1][1:top_n+1]
    
    results = [(feature_names[i], similarities[i]) for i in top_indices]
    return results

# Test with some words
test_words = ['computer', 'space', 'hockey', 'medical', 'government']

for word in test_words:
    print(f"\nWords similar to '{word}' (LSA):")
    similar = find_similar_words_lsa(word, word_embeddings_lsa, feature_names, top_n=8)
    if isinstance(similar, str):
        print(f"  {similar}")
    else:
        for sim_word, score in similar:
            print(f"  {sim_word:<15} {score:.4f}")

### 💡 Exercise 1: Exploring LSA

1. Try different numbers of components (50, 100, 200). How does it affect the results?
2. Find similar words for domain-specific terms. Do the results make sense?
3. What are the limitations of LSA that you observe?

In [ ]:
# Your experiments here


## Part 3: Topic Modeling - LDA (Latent Dirichlet Allocation)

LDA is a **probabilistic topic model**:
- Each document is a mixture of topics
- Each topic is a distribution over words
- Different from LSA - models generative process

While not strictly an embedding method, LDA creates interpretable topic representations.

In [ ]:
# Create count matrix (LDA uses counts, not TF-IDF)
count_vectorizer = CountVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.5,
    stop_words='english'
)

count_matrix = count_vectorizer.fit_transform(documents)
count_feature_names = count_vectorizer.get_feature_names_out()

print(f"Count matrix shape: {count_matrix.shape}")

In [ ]:
# Fit LDA model
n_topics = 8  # Same as number of categories

print(f"Fitting LDA with {n_topics} topics...")
lda = LatentDirichletAllocation(
    n_components=n_topics,
    max_iter=20,
    learning_method='online',
    random_state=42,
    n_jobs=-1
)

doc_topic_dist = lda.fit_transform(count_matrix)

print(f"\n✓ Document-topic distribution shape: {doc_topic_dist.shape}")
print(f"✓ Each document is represented as distribution over {n_topics} topics")

In [ ]:
# Display top words for each topic
def display_topics(model, feature_names, n_top_words=10):
    """
    Display top words for each topic in LDA model.
    """
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        print(f"\nTopic {topic_idx}:")
        print(f"  {', '.join(top_words)}")

print("Top words per topic:")
print("=" * 80)
display_topics(lda, count_feature_names, n_top_words=12)

In [ ]:
# Examine document-topic distributions
def show_doc_topics(doc_idx, doc_topic_dist, doc_text, n_chars=200):
    """
    Show topic distribution for a document.
    """
    print(f"Document {doc_idx}:")
    print(f"Text: {doc_text[:n_chars]}...\n")
    
    dist = doc_topic_dist[doc_idx]
    top_topics = np.argsort(dist)[::-1][:3]
    
    print("Topic distribution:")
    for topic_id in top_topics:
        print(f"  Topic {topic_id}: {dist[topic_id]:.3f}")

# Show examples
for i in [0, 100, 500]:
    print("\n" + "=" * 80)
    show_doc_topics(i, doc_topic_dist, documents[i])
    print()

### 💡 Exercise 2: Interpreting Topics

1. Can you identify what each topic represents based on the top words?
2. Do the topics align with the known categories?
3. What happens if you change the number of topics?

**Your analysis:**

Topic interpretations:
- Topic 0: 
- Topic 1: 
- ...

---

## Part 4: Word2Vec - Neural Word Embeddings

Word2Vec learns embeddings by predicting context:
- **Skip-gram**: Predict context words from target word
- **CBOW**: Predict target word from context

Key idea: Words that appear in similar contexts should have similar embeddings.

In [ ]:
# Train Word2Vec model
print("Training Word2Vec model...")
print(f"Using {len(tokenized_docs)} documents with {len(vocab)} unique words\n")

# Parameters
w2v_model = Word2Vec(
    sentences=tokenized_docs,
    vector_size=100,      # Embedding dimension
    window=5,             # Context window size
    min_count=5,          # Minimum word frequency
    workers=4,            # Parallel training
    sg=1,                 # Skip-gram (sg=1) vs CBOW (sg=0)
    epochs=10,
    seed=42
)

print(f"✓ Model trained!")
print(f"✓ Vocabulary size: {len(w2v_model.wv)}")
print(f"✓ Embedding dimension: {w2v_model.wv.vector_size}")

In [ ]:
# Test word similarities
def find_similar_words_w2v(word, model, top_n=10):
    """
    Find most similar words using Word2Vec.
    """
    try:
        similar = model.wv.most_similar(word, topn=top_n)
        return similar
    except KeyError:
        return f"Word '{word}' not in vocabulary"

# Test with various words
test_words = ['computer', 'space', 'hockey', 'doctor', 'government', 'team', 'nasa']

for word in test_words:
    print(f"\nWords similar to '{word}' (Word2Vec):")
    similar = find_similar_words_w2v(word, w2v_model, top_n=8)
    if isinstance(similar, str):
        print(f"  {similar}")
    else:
        for sim_word, score in similar:
            print(f"  {sim_word:<15} {score:.4f}")

In [ ]:
# Word analogies: A is to B as C is to D
def word_analogy(w2v, word1, word2, word3, top_n=5):
    """
    Solve analogy: word1 is to word2 as word3 is to ?
    """
    try:
        result = w2v.wv.most_similar(positive=[word3, word2], negative=[word1], topn=top_n)
        return result
    except KeyError as e:
        return f"Error: {e}"

# Test analogies
analogies = [
    ('man', 'woman', 'king'),    # Should give 'queen'
    ('baseball', 'bat', 'hockey'),  # Should give 'stick' or 'puck'
    ('doctor', 'hospital', 'teacher'),  # Should give 'school'
]

print("Word Analogies:")
print("=" * 80)
for w1, w2, w3 in analogies:
    print(f"\n'{w1}' is to '{w2}' as '{w3}' is to:")
    result = word_analogy(w2v_model, w1, w2, w3, top_n=5)
    if isinstance(result, str):
        print(f"  {result}")
    else:
        for word, score in result:
            print(f"  {word:<15} {score:.4f}")

### 💡 Exercise 3: Word2Vec Exploration

1. Create your own word analogies. What works? What doesn't?
2. Compare Word2Vec similarities to LSA similarities. Which seems more semantic?
3. Try training with CBOW instead of Skip-gram (`sg=0`). How do results differ?

In [ ]:
# Your experiments here

# Example: Train CBOW model
# w2v_cbow = Word2Vec(
#     sentences=tokenized_docs,
#     vector_size=100,
#     window=5,
#     min_count=5,
#     sg=0,  # CBOW
#     epochs=10,
#     seed=42
# )


## Part 5: Visualizing Embeddings with UMAP

Embeddings are high-dimensional (50-300 dims). To visualize them, we need to project to 2D.

**UMAP** (Uniform Manifold Approximation and Projection):
- Better than t-SNE for preserving global structure
- Faster than t-SNE
- Preserves both local and global structure

In [ ]:
# Get word vectors from Word2Vec
words = list(w2v_model.wv.index_to_key)[:1000]  # Use top 1000 words
word_vectors = np.array([w2v_model.wv[word] for word in words])

print(f"Visualizing {len(words)} words")
print(f"Original dimension: {word_vectors.shape[1]}")

In [ ]:
# Apply UMAP
print("Applying UMAP reduction to 2D...")
umap_reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    metric='cosine',
    random_state=42
)

word_vectors_2d = umap_reducer.fit_transform(word_vectors)
print(f"✓ Reduced to 2D: {word_vectors_2d.shape}")

In [ ]:
# Visualize all words
plt.figure(figsize=(14, 10))
plt.scatter(word_vectors_2d[:, 0], word_vectors_2d[:, 1], alpha=0.3, s=10)
plt.title('Word2Vec Embeddings Visualized with UMAP', fontsize=16)
plt.xlabel('UMAP Dimension 1')
plt.ylabel('UMAP Dimension 2')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Highlight specific semantic clusters
def visualize_word_clusters(words_2d, words_list, word_groups, title='Word Clusters'):
    """
    Visualize specific groups of words in the embedding space.
    
    Args:
        words_2d: 2D coordinates of all words
        words_list: List of all words
        word_groups: Dict of {group_name: [word1, word2, ...]}
    """
    plt.figure(figsize=(14, 10))
    
    # Plot all words in gray
    plt.scatter(words_2d[:, 0], words_2d[:, 1], alpha=0.1, s=5, c='gray')
    
    # Plot each group in different color
    colors = plt.cm.tab10(np.linspace(0, 1, len(word_groups)))
    
    for (group_name, group_words), color in zip(word_groups.items(), colors):
        # Find indices of words in this group
        indices = [i for i, w in enumerate(words_list) if w in group_words]
        
        if indices:
            group_coords = words_2d[indices]
            plt.scatter(group_coords[:, 0], group_coords[:, 1], 
                       alpha=0.7, s=100, c=[color], label=group_name)
            
            # Annotate words
            for idx in indices:
                plt.annotate(words_list[idx], 
                           (words_2d[idx, 0], words_2d[idx, 1]),
                           fontsize=9, alpha=0.8)
    
    plt.title(title, fontsize=16)
    plt.xlabel('UMAP Dimension 1')
    plt.ylabel('UMAP Dimension 2')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Define semantic clusters to visualize
word_clusters = {
    'Sports': ['hockey', 'baseball', 'player', 'team', 'game', 'season'],
    'Space': ['space', 'nasa', 'launch', 'orbit', 'moon', 'shuttle'],
    'Computers': ['computer', 'software', 'program', 'file', 'system', 'windows'],
    'Medicine': ['medical', 'doctor', 'patient', 'hospital', 'treatment', 'disease'],
    'Politics': ['government', 'president', 'political', 'congress', 'law']
}

visualize_word_clusters(word_vectors_2d, words, word_clusters, 
                       title='Semantic Clusters in Word2Vec Embeddings')

### 💡 Exercise 4: Cluster Analysis

1. Do related words cluster together in the visualization?
2. Can you find interesting semantic relationships in the 2D space?
3. Try defining your own word groups and visualizing them.

In [ ]:
# Your custom word clusters
custom_clusters = {
    # Define your own clusters here
}

# visualize_word_clusters(word_vectors_2d, words, custom_clusters)


## Part 6: Comparing Embedding Methods

Let's quantitatively compare LSA, LDA, and Word2Vec.

In [ ]:
# Compare similarity rankings for the same word across methods
def compare_similarities(word, lsa_sim, w2v_sim):
    """
    Compare similarity rankings across methods.
    """
    print(f"\nSimilar words for '{word}':")
    print("=" * 80)
    print(f"{'LSA':<40} {'Word2Vec':<40}")
    print("-" * 80)
    
    max_len = max(len(lsa_sim), len(w2v_sim))
    for i in range(min(10, max_len)):
        lsa_str = f"{lsa_sim[i][0]:<25} ({lsa_sim[i][1]:.3f})" if i < len(lsa_sim) else ""
        w2v_str = f"{w2v_sim[i][0]:<25} ({w2v_sim[i][1]:.3f})" if i < len(w2v_sim) else ""
        print(f"{lsa_str:<40} {w2v_str:<40}")

# Compare for several words
comparison_words = ['computer', 'hockey', 'space']

for word in comparison_words:
    lsa_sim = find_similar_words_lsa(word, word_embeddings_lsa, feature_names, top_n=10)
    w2v_sim = find_similar_words_w2v(word, w2v_model, top_n=10)
    
    if not isinstance(lsa_sim, str) and not isinstance(w2v_sim, str):
        compare_similarities(word, lsa_sim, w2v_sim)

### Intrinsic Evaluation: Word Similarity

We can evaluate embeddings on word similarity benchmarks.

In [ ]:
# Create a simple word similarity test
word_pairs = [
    # (word1, word2, expected_similarity_category)
    ('computer', 'software', 'high'),
    ('computer', 'graphics', 'high'),
    ('hockey', 'baseball', 'high'),
    ('hockey', 'player', 'high'),
    ('space', 'nasa', 'high'),
    ('computer', 'hockey', 'low'),
    ('space', 'medical', 'low'),
    ('government', 'graphics', 'low'),
]

def evaluate_similarities(model, pairs):
    """
    Evaluate word pair similarities.
    """
    results = []
    
    for w1, w2, expected in pairs:
        try:
            similarity = model.wv.similarity(w1, w2)
            results.append((w1, w2, similarity, expected))
        except KeyError:
            results.append((w1, w2, None, expected))
    
    return results

# Evaluate
results = evaluate_similarities(w2v_model, word_pairs)

print("Word Pair Similarities (Word2Vec):")
print("=" * 70)
print(f"{'Word 1':<15} {'Word 2':<15} {'Similarity':<12} {'Expected'}")
print("-" * 70)

for w1, w2, sim, expected in results:
    sim_str = f"{sim:.4f}" if sim is not None else "N/A"
    print(f"{w1:<15} {w2:<15} {sim_str:<12} {expected}")

# Calculate correlation
high_sims = [sim for _, _, sim, exp in results if exp == 'high' and sim is not None]
low_sims = [sim for _, _, sim, exp in results if exp == 'low' and sim is not None]

print(f"\nAverage similarity for 'high' pairs: {np.mean(high_sims):.4f}")
print(f"Average similarity for 'low' pairs: {np.mean(low_sims):.4f}")
print(f"Separation: {np.mean(high_sims) - np.mean(low_sims):.4f}")

## Part 7: Document Classification with Embeddings

Let's use embeddings for classification and compare to last week's TF-IDF baseline.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Split data
X_train_docs, X_test_docs, y_train, y_test = train_test_split(
    documents, labels, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train_docs)} documents")
print(f"Test set: {len(X_test_docs)} documents")

In [ ]:
# Method 1: Average Word2Vec embeddings
def document_vector_w2v(doc, model):
    """
    Compute document vector by averaging word vectors.
    """
    tokens = preprocess_for_embeddings(doc)
    # Get vectors for words in vocabulary
    word_vecs = [model.wv[word] for word in tokens if word in model.wv]
    
    if len(word_vecs) == 0:
        return np.zeros(model.wv.vector_size)
    
    return np.mean(word_vecs, axis=0)

# Create document vectors
print("Creating document vectors with Word2Vec...")
X_train_w2v = np.array([document_vector_w2v(doc, w2v_model) for doc in X_train_docs])
X_test_w2v = np.array([document_vector_w2v(doc, w2v_model) for doc in X_test_docs])

print(f"Training features shape: {X_train_w2v.shape}")
print(f"Test features shape: {X_test_w2v.shape}")

In [ ]:
# Train classifier on Word2Vec embeddings
print("Training classifier on Word2Vec embeddings...")
clf_w2v = LogisticRegression(max_iter=1000, random_state=42)
clf_w2v.fit(X_train_w2v, y_train)

y_pred_w2v = clf_w2v.predict(X_test_w2v)
acc_w2v = accuracy_score(y_test, y_pred_w2v)

print(f"\n✓ Accuracy: {acc_w2v:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_w2v, target_names=label_names))

In [ ]:
# Compare with TF-IDF baseline
print("Training TF-IDF baseline...")
tfidf_clf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf_clf = tfidf_clf.fit_transform(X_train_docs)
X_test_tfidf_clf = tfidf_clf.transform(X_test_docs)

clf_tfidf = LogisticRegression(max_iter=1000, random_state=42)
clf_tfidf.fit(X_train_tfidf_clf, y_train)

y_pred_tfidf = clf_tfidf.predict(X_test_tfidf_clf)
acc_tfidf = accuracy_score(y_test, y_pred_tfidf)

print(f"\n✓ TF-IDF Accuracy: {acc_tfidf:.4f}")
print(f"✓ Word2Vec Accuracy: {acc_w2v:.4f}")
print(f"\nDifference: {acc_w2v - acc_tfidf:+.4f}")

### 💡 Exercise 5: Improving Document Representations

The simple average of word vectors may not be optimal. Try:

1. **TF-IDF weighted averaging**: Weight word vectors by their TF-IDF scores
2. **Doc2Vec**: Use Gensim's Doc2Vec for direct document embeddings
3. **Max pooling**: Take max instead of mean

Can you beat the TF-IDF baseline?

In [ ]:
# Your improved document representation here

# Example: TF-IDF weighted averaging
# def document_vector_tfidf_weighted(doc, w2v_model, tfidf_vectorizer, feature_names):
#     tokens = preprocess_for_embeddings(doc)
#     # Get TF-IDF scores for document
#     # Weight word vectors by TF-IDF
#     # Return weighted average
#     pass


## Part 8: Discussion Questions

Discuss with your group:

1. **LSA vs Word2Vec**: What are the key differences? When would you use each?

2. **Embedding Quality**: How do you know if embeddings are "good"? What makes one embedding better than another?

3. **Dimensionality**: Why use 100-300 dimensions? What happens with too few? Too many?

4. **Context Windows**: In Word2Vec, how does window size affect the embeddings learned?

5. **Out-of-Vocabulary**: How do you handle words not in your embedding vocabulary?

6. **Bias in Embeddings**: Embeddings learn from text data. What biases might they capture? Why is this concerning?

7. **Modern Embeddings**: How do contextualized embeddings (BERT, GPT) differ from Word2Vec?

**Your notes:**

---

(Discussion notes here)

---

## Summary and Key Takeaways

Today you learned:

1. ✅ **LSA/SVD**: Dimensionality reduction on TF-IDF matrices
2. ✅ **LDA**: Probabilistic topic modeling
3. ✅ **Word2Vec**: Neural word embeddings from context
4. ✅ **UMAP**: Visualizing high-dimensional embeddings
5. ✅ **Evaluation**: Intrinsic (similarity) and extrinsic (classification) evaluation

### Key Insights:

**Embeddings capture semantic meaning:**
- Similar words have similar vectors
- Can perform vector arithmetic (king - man + woman ≈ queen)
- Clusters emerge naturally in embedding space

**Different methods have different strengths:**
- **LSA**: Fast, interpretable, works on smaller data
- **LDA**: Interpretable topics, probabilistic framework
- **Word2Vec**: Better semantic quality, needs more data

**Limitations to consider:**
- Static embeddings (one vector per word, no context)
- Requires large amounts of text data
- Can encode biases from training data
- Out-of-vocabulary problem

### Looking Ahead:

Modern NLP uses **contextualized embeddings**:
- BERT, GPT, etc. create different embeddings based on context
- "bank" in "river bank" vs "bank account" gets different vectors
- We'll explore these in later weeks!

---

**Questions? Office hours or Piazza!**